In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller
import statsmodels.api as sm
from arch import arch_model
from arch.unitroot import PhillipsPerron
from mgarch import mgarch

In [49]:
from garch_functions import (
    fit_all_garch,
    diagnostics_summary_table,
    parameter_significance_table,
    plot_all_diagnostics,
    GARCHConfig,
    DEFAULT_CONFIG
)

df = pd.read_csv("data/merged_data_v2.csv", index_col='date')
df.dropna(inplace=True)
df.index= pd.to_datetime(df.index, format = '%Y-%m-%d')
weekly_df = df[["btc_adj_close", "eth_adj_close", "sp500_adj_close", "vti_adj_close", "agg_adj_close", "tlt_adj_close"]].resample("W-FRI").last().dropna()
weekly_df.columns = ["BTC", "ETH", "SP500", "VTI", "AGG", "TLT"]
price_assets = ["BTC", "ETH", "SP500", "VTI", "AGG", "TLT"] # treasury yield is in rates, will be treated differently
weekly_logret = 100*np.log(weekly_df[price_assets] / weekly_df[price_assets].shift(1)).dropna()

garch_results = fit_all_garch(weekly_logret, model_type="auto")
diag_table = diagnostics_summary_table(garch_results)
cond_vol_df = pd.DataFrame({
    asset: garch_results[asset]["cond_vol"]
    for asset in garch_results
}).dropna()


GARCH fit: BTC

  --------------------------------------------------
  AR(1) mean detected (LB p=0.0843) -- using AR(1) mean equation

  Model competition [primary=skewt, mean=AR]:
  Model            Dist            AIC        BIC     Status
  --------------------------------------------------------
  GARCH            skewt       3019.33    3047.68         ok
  GJR              skewt       3013.32    3045.72         ok
  EGARCH           skewt            --         --   no convergence -> retrying with GED
  EGARCH           ged         2996.36    3024.71 GED fallback

  -> BIC selects: EGARCH-GED  (BIC=3024.71)
  EGARCH |beta|=0.9966  ->  stationary  |  near-IGARCH -- shocks near-permanent (half-life ~205 weeks)

  Post-fit [EGARCH-GED]:
  Ljung-Box resid   p@10=0.7343  p@20=0.6759
  Ljung-Box resid^2 p@10=0.3206  p@20=0.2469
  ARCH LM [std residuals -- post EGARCH]  LM=12.3459  p=0.4183  F=1.0271  p=0.4226  ->  no ARCH effects

  Normality / tail check [BTC]
  Obs                    

                                 Mean Model                                
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
Const          0.2544      0.105      2.416  1.570e-02  [4.802e-02,  0.461]
VTI[1]        -0.1461  5.075e-02     -2.878  4.002e-03 [ -0.246,-4.659e-02]

GARCH fit: AGG

  --------------------------------------------------
  No serial correlation (LB p=0.2918) -- using Constant mean

  Model competition [primary=skewt, mean=Constant]:
  Model            Dist            AIC        BIC     Status
  --------------------------------------------------------
  GARCH            skewt        834.80     859.11         ok
  GJR              skewt        836.79     865.15         ok
  EGARCH           skewt        833.02     861.38         ok

  -> BIC selects: GARCH-SKEWT  (BIC=859.11)
  GARCH alpha+beta=0.9768  ->  stationary  |  highly persistent (half-life ~30 weeks)

 

In [50]:
#Check Stationarity (ADF)
from statsmodels.tsa.stattools import adfuller
import pandas as pd

results = []

for col in cond_vol_df.columns:
    adf_result = adfuller(cond_vol_df[col])

    test_stat = adf_result[0]
    p_value = adf_result[1]

    if p_value < 0.01:
        sig = "***"
    elif p_value < 0.05:
        sig = "**"
    elif p_value < 0.10:
        sig = "*"
    else:
        sig = ""

    results.append([col, test_stat, p_value, sig])

df_adf = pd.DataFrame(results,
                      columns=["Asset", "ADF Statistic", "p-value", "Sig"])

df_adf["ADF Statistic"] = df_adf["ADF Statistic"].round(4)
df_adf["p-value"] = df_adf["p-value"].round(6)

df_adf

,Asset,ADF Statistic,p-value,Sig
0,BTC,-2.0334,0.272112,
1,ETH,-2.2285,0.196062,
2,SP500,-6.2239,0.000000,***
3,VTI,-6.2222,0.000000,***
4,AGG,-5.4366,0.000003,***
5,TLT,-4.0378,0.001225,***


In [51]:
cond_vol_df = np.log(cond_vol_df).diff().dropna() * 100
cond_vol_df

,BTC,ETH,SP500,VTI,AGG,TLT
date,,,,,,
2017-12-01,1.170181,-0.656657,-8.777444,-8.877209,-2.831458,-0.299449
2017-12-08,1.044101,1.081419,-9.123422,-8.369848,-1.348823,-1.146399
2017-12-15,0.976502,0.950805,-6.808426,-7.114403,-4.023888,-0.916768
2017-12-22,1.166562,-0.977887,-6.854497,-6.672631,1.388773,5.189985
2017-12-29,-6.036601,1.174962,-5.337196,-5.960520,22.787543,14.530895
...,...,...,...,...,...,...
2025-12-05,1.477643,1.090894,-12.282077,-10.586649,-1.400118,-0.484049
2025-12-12,-0.562176,1.776116,-8.197910,-8.637170,7.904763,5.639117
2025-12-19,1.582434,2.413887,3.455060,0.662460,-5.110166,-0.584695


In [36]:
# edit to the above, remove if necessary
# BTC and ETH → log returns
for col in ['BTC', 'ETH']:
    cond_vol_df[col] = np.log(cond_vol_df[col]).diff()

# Scale BTC and ETH log returns to % change
cond_vol_df['BTC'] = cond_vol_df['BTC'] * 100
cond_vol_df['ETH'] = cond_vol_df['ETH'] * 100

# Only drop rows where BTC/ETH diff created NaN
cond_vol_df.dropna(subset=['BTC', 'ETH'], inplace=True)

cond_vol_df

,BTC,ETH,SP500,VTI,AGG,TLT
date,,,,,,
2017-12-01,1.170181,-0.656657,1.974381,1.938315,0.378420,1.283179
2017-12-08,1.044101,1.081419,1.802223,1.782685,0.373350,1.268553
2017-12-15,0.976502,0.950805,1.683604,1.660264,0.358625,1.256976
2017-12-22,1.166562,-0.977887,1.572068,1.553096,0.363641,1.323936
2017-12-29,-6.036601,1.174962,1.490363,1.463228,0.456707,1.530995
...,...,...,...,...,...,...
2025-12-05,1.477643,1.090894,2.020118,2.079538,0.463404,1.437077
2025-12-12,-0.562176,1.776116,1.861117,1.907463,0.501521,1.520444
2025-12-19,1.582434,2.413887,1.926543,1.920141,0.476537,1.511580


In [52]:
#Check Stationarity (ADF)
from statsmodels.tsa.stattools import adfuller
import pandas as pd

results = []

for col in cond_vol_df.columns:
    adf_result = adfuller(cond_vol_df[col])

    test_stat = adf_result[0]
    p_value = adf_result[1]

    if p_value < 0.01:
        sig = "***"
    elif p_value < 0.05:
        sig = "**"
    elif p_value < 0.10:
        sig = "*"
    else:
        sig = ""

    results.append([col, test_stat, p_value, sig])

df_adf = pd.DataFrame(results,
                      columns=["Asset", "ADF Statistic", "p-value", "Sig"])

df_adf["ADF Statistic"] = df_adf["ADF Statistic"].round(4)
df_adf["p-value"] = df_adf["p-value"].round(6)

df_adf

,Asset,ADF Statistic,p-value,Sig
0,BTC,-21.2452,0.0,***
1,ETH,-22.0682,0.0,***
2,SP500,-19.7265,0.0,***
3,VTI,-19.6461,0.0,***
4,AGG,-13.3039,0.0,***
5,TLT,-13.0788,0.0,***


In [53]:
#VECTOR AUTOREGRESSION (VAR)
cond_vol_df.index = pd.to_datetime(cond_vol_df.index)

from statsmodels.tsa.api import VAR

model = VAR(cond_vol_df)
lag_selection = model.select_order(maxlags=8)
print(lag_selection.summary())

results = model.fit(lag_selection.aic)
print(results.summary())

 VAR Order Selection (* highlights the minimums) 
      AIC         BIC         FPE         HQIC   
-------------------------------------------------
0      18.13*      18.19*  7.466e+07*      18.15*
1       18.17       18.57   7.756e+07       18.33
2       18.21       18.97   8.116e+07       18.51
3       18.27       19.38   8.644e+07       18.71
4       18.36       19.81   9.408e+07       18.93
5       18.46       20.27   1.044e+08       19.18
6       18.51       20.67   1.100e+08       19.36
7       18.59       21.09   1.187e+08       19.58
8       18.60       21.46   1.209e+08       19.73
-------------------------------------------------
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Sun, 29, Mar, 2026
Time:                     11:43:57
--------------------------------------------------------------------
No. of Equations:         6.00000    BIC:                    18.1917
Nobs:                     423.000    

In [54]:
#GRANGER CAUSALITY TEST
from statsmodels.tsa.stattools import grangercausalitytests
import itertools
import pandas as pd

max_lag = 1 #BIC
alpha = 0.05

results_list = []

columns = cond_vol_df.columns

for y_col, x_col in itertools.permutations(columns, 2):
    test_result = grangercausalitytests(cond_vol_df[[y_col, x_col]], maxlag=max_lag, verbose=False)

    f_stat = test_result[max_lag][0]['ssr_ftest'][0]
    p_value = test_result[max_lag][0]['ssr_ftest'][1]

    # significance stars
    if p_value < 0.01:
        sig = "***"
    elif p_value < 0.05:
        sig = "**"
    elif p_value < 0.10:
        sig = "*"
    else:
        sig = ""

    results_list.append([x_col, y_col, f_stat, p_value, sig])

df_results = pd.DataFrame(results_list, columns=["Cause", "Target", "F-stat", "p-value", "Sig"])

print(df_results)

    Cause Target    F-stat   p-value  Sig
0     ETH    BTC  0.923435  0.337129     
1   SP500    BTC  0.258410  0.611482     
2     VTI    BTC  0.239802  0.624606     
3     AGG    BTC  0.202110  0.653255     
4     TLT    BTC  0.632342  0.426948     
5     BTC    ETH  2.596060  0.107883     
6   SP500    ETH  2.390865  0.122801     
7     VTI    ETH  2.404734  0.121724     
8     AGG    ETH  1.554399  0.213184     
9     TLT    ETH  4.325285  0.038158   **
10    BTC  SP500  2.881731  0.090332    *
11    ETH  SP500  0.297804  0.585553     
12    VTI  SP500  0.286782  0.592575     
13    AGG  SP500  1.863077  0.173003     
14    TLT  SP500  2.398302  0.122222     
15    BTC    VTI  2.257859  0.133691     
16    ETH    VTI  0.174172  0.676644     
17  SP500    VTI  0.634031  0.426331     
18    AGG    VTI  1.274233  0.259620     
19    TLT    VTI  1.859838  0.173376     
20    BTC    AGG  1.438653  0.231035     
21    ETH    AGG  0.536895  0.464132     
22  SP500    AGG  5.524679  0.0192

/Users/junrongng/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1488: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/junrongng/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1488: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/junrongng/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1488: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/junrongng/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1488: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/junrongng/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1488: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/junrongng/anaconda3/lib/python3.11/site-packages/statsmodels/ts

In [47]:
from statsmodels.tsa.stattools import grangercausalitytests
import itertools
import pandas as pd

max_lag = 3 #FPE & AIC
alpha = 0.05

results_list = []

columns = cond_vol_df.columns

for y_col, x_col in itertools.permutations(columns, 2):
    test_result = grangercausalitytests(cond_vol_df[[y_col, x_col]], maxlag=max_lag, verbose=False)

    f_stat = test_result[max_lag][0]['ssr_ftest'][0]
    p_value = test_result[max_lag][0]['ssr_ftest'][1]

    # significance stars
    if p_value < 0.01:
        sig = "***"
    elif p_value < 0.05:
        sig = "**"
    elif p_value < 0.10:
        sig = "*"
    else:
        sig = ""

    results_list.append([x_col, y_col, f_stat, p_value, sig])

df_results = pd.DataFrame(results_list, columns=["Cause", "Target", "F-stat", "p-value", "Sig"])

print(df_results)

    Cause Target    F-stat   p-value  Sig
0     ETH    BTC  0.385366  0.763604     
1   SP500    BTC  3.783618  0.010637   **
2     VTI    BTC  3.994274  0.008002  ***
3     AGG    BTC  1.657830  0.175514     
4     TLT    BTC  1.641909  0.179077     
5     BTC    ETH  1.475135  0.220720     
6   SP500    ETH  2.537367  0.056240    *
7     VTI    ETH  2.675460  0.046867   **
8     AGG    ETH  1.249970  0.291240     
9     TLT    ETH  1.954784  0.120173     
10    BTC  SP500  1.110814  0.344443     
11    ETH  SP500  0.374827  0.771209     
12    VTI  SP500  0.360856  0.781310     
13    AGG  SP500  1.310453  0.270512     
14    TLT  SP500  1.146857  0.329894     
15    BTC    VTI  0.932470  0.424926     
16    ETH    VTI  0.347930  0.790671     
17  SP500    VTI  0.350682  0.788677     
18    AGG    VTI  0.914559  0.433823     
19    TLT    VTI  0.842763  0.471036     
20    BTC    AGG  0.465852  0.706266     
21    ETH    AGG  1.152489  0.327670     
22  SP500    AGG  6.101243  0.0004

/Users/junrongng/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1488: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/junrongng/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1488: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/junrongng/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1488: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/junrongng/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1488: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/junrongng/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1488: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/junrongng/anaconda3/lib/python3.11/site-packages/statsmodels/ts

In [55]:
from statsmodels.tsa.stattools import grangercausalitytests
import itertools

max_lag = 4
alpha = 0.05

# Loop through all combinations of columns (Y, X) where Y != X
columns = cond_vol_df.columns
for y_col, x_col in itertools.permutations(columns, 2):
    print(f"\nTesting if '{x_col}' Granger-causes '{y_col}':")
    results = grangercausalitytests(cond_vol_df[[y_col, x_col]], maxlag=max_lag, verbose=False)

    for lag in range(1, max_lag + 1):
        f_test_p = results[lag][0]['ssr_ftest'][1]  # p-value of SSR F-test
        significance = "Significant" if f_test_p < alpha else "Not significant"
        symbol = "<" if f_test_p < alpha else ">"
        print(f"  Lag {lag}: p = {f_test_p:.4f} {symbol} {alpha} → {significance}")


Testing if 'ETH' Granger-causes 'BTC':
  Lag 1: p = 0.3371 > 0.05 → Not significant
  Lag 2: p = 0.5565 > 0.05 → Not significant
  Lag 3: p = 0.7636 > 0.05 → Not significant
  Lag 4: p = 0.8435 > 0.05 → Not significant

Testing if 'SP500' Granger-causes 'BTC':
  Lag 1: p = 0.6115 > 0.05 → Not significant
  Lag 2: p = 0.0314 < 0.05 → Significant
  Lag 3: p = 0.0106 < 0.05 → Significant
  Lag 4: p = 0.0173 < 0.05 → Significant

Testing if 'VTI' Granger-causes 'BTC':
  Lag 1: p = 0.6246 > 0.05 → Not significant
  Lag 2: p = 0.0284 < 0.05 → Significant
  Lag 3: p = 0.0080 < 0.05 → Significant
  Lag 4: p = 0.0138 < 0.05 → Significant

Testing if 'AGG' Granger-causes 'BTC':
  Lag 1: p = 0.6533 > 0.05 → Not significant
  Lag 2: p = 0.1413 > 0.05 → Not significant
  Lag 3: p = 0.1755 > 0.05 → Not significant
  Lag 4: p = 0.2131 > 0.05 → Not significant

Testing if 'TLT' Granger-causes 'BTC':
  Lag 1: p = 0.4269 > 0.05 → Not significant
  Lag 2: p = 0.0987 > 0.05 → Not significant
  Lag 3: p =

/Users/junrongng/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1488: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/junrongng/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1488: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/junrongng/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1488: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/junrongng/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1488: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/junrongng/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1488: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/junrongng/anaconda3/lib/python3.11/site-packages/statsmodels/ts

## Importing Bubble Indicators

In [66]:
import pickle
with open("gsadf_results.pkl", "rb") as f:
    results = pickle.load(f)

In [71]:
results.keys()

dict_keys(['BTC', 'ETH', 'S&P500'])

In [73]:
results["BTC"]['bubbles']

[(datetime.date(2017, 1, 22), datetime.date(2017, 3, 5)),
 (datetime.date(2017, 4, 23), datetime.date(2018, 5, 20)),
 (datetime.date(2021, 1, 3), datetime.date(2021, 5, 16))]

## VAR with Interaction terms